# 03. Deep Learning Optimizers (Momentum, RMSprop, Adam & AdamW)

**The evolution of gradient-based optimization: From Stochastic Gradient Descent to AdamW and modern Learning Rate Schedulers.**

---

## 1. The Optimization Landscape in Deep Learning

In Deep Learning, we minimize an empirical risk over millions of training samples:
$$\min_{\mathbf{w}} L(\mathbf{w}) = \frac{1}{N} \sum_{i=1}^N \ell(f(\mathbf{x}_i; \mathbf{w}), y_i)$$

- **Batch Gradient Descent**: Computes the gradient over all $N$ training samples. Exact, but intractable for large datasets ($O(N)$ memory/compute).
- **Stochastic Gradient Descent (SGD)**: Computes the gradient on a single random sample $i$. Very fast, but noisy gradient steps.
- **Mini-Batch SGD**: Computes the gradient over a mini-batch $\mathcal{B}$ of size $B \in [32, 512]$:
  $$\mathbf{g}_t = \frac{1}{B} \sum_{i \in \mathcal{B}} \nabla_{\mathbf{w}} \ell_i(\mathbf{w}_t), \quad \mathbf{w}_{t+1} = \mathbf{w}_t - \eta \mathbf{g}_t$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy and Matplotlib loaded!")


---

## 2. Momentum & Nesterov Accelerated Gradient (NAG)

### The Problem with standard SGD:
In ill-conditioned ravines (steep in one dimension, shallow in another), standard SGD oscillates wildly across the steep direction while making slow progress along the flat valley.

### SGD with Momentum (Polyak, 1964):
Maintains a velocity vector $\mathbf{v}_t$ (like a heavy ball rolling downhill):
$$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \eta \mathbf{g}_t \quad (\beta \approx 0.9)$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{v}_t$$

- **Effect**: Accelerates progress along consistent gradient directions while canceling out perpendicular oscillations.


In [ ]:
# Simulating SGD vs Momentum on an elongated ravine: f(x, y) = 0.1*x^2 + 2.0*y^2
def ravine_loss(x, y): return 0.1*x**2 + 2.0*y**2
def ravine_grad(x, y): return np.array([0.2*x, 4.0*y])

# 1. Standard SGD
w_sgd = np.array([5.0, 3.0])
lr = 0.35
sgd_path = [w_sgd.copy()]
for _ in range(30):
    w_sgd -= lr * ravine_grad(w_sgd[0], w_sgd[1])
    sgd_path.append(w_sgd.copy())
sgd_path = np.array(sgd_path)

# 2. Momentum
w_mom = np.array([5.0, 3.0])
v = np.zeros(2)
beta = 0.85
mom_path = [w_mom.copy()]
for _ in range(30):
    v = beta * v + lr * ravine_grad(w_mom[0], w_mom[1])
    w_mom -= v
    mom_path.append(w_mom.copy())
mom_path = np.array(mom_path)

# Plot trajectories
x_grid = np.linspace(-6, 6, 200)
y_grid = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = ravine_loss(X, Y)

plt.figure(figsize=(10, 5))
plt.contour(X, Y, Z, levels=20, cmap='gray_r', alpha=0.6)
plt.plot(sgd_path[:, 0], sgd_path[:, 1], 'r.-', linewidth=1.5, label='Standard SGD (Oscillating)')
plt.plot(mom_path[:, 0], mom_path[:, 1], 'b.-', linewidth=2, label='SGD + Momentum (Fast & Smooth)')
plt.title("SGD vs Momentum on an Ill-Conditioned Ravine")
plt.xlabel("x"); plt.ylabel("y")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---

## 3. Adaptive Learning Rate Optimizers (AdaGrad & RMSprop)

Instead of using the same learning rate for all parameters, adapt the learning rate per parameter based on historical gradient magnitudes.

### RMSprop (Hinton, 2012):
Keeps an exponentially decaying average of squared gradients $\mathbf{s}_t$:
$$\mathbf{s}_t = \beta \mathbf{s}_{t-1} + (1 - \beta) \mathbf{g}_t^2 \quad (\beta \approx 0.99)$$
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t$$

- Large gradients $\implies$ smaller step size (prevents explosion).
- Small gradients $\implies$ larger step size (accelerates learning in flat regions).


---

## 4. Adam (Adaptive Moment Estimation)

**Adam (Kingma & Ba, 2014)** combines the best of **Momentum** (1st moment $m_t$) and **RMSprop** (2nd moment $v_t$) with bias corrections:

1. Update 1st moment (mean): $\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1) \mathbf{g}_t$ ($\beta_1 = 0.9$)
2. Update 2nd moment (uncentered variance): $\mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2) \mathbf{g}_t^2$ ($\beta_2 = 0.999$)
3. **Bias Corrections** (critical for early steps when $m_0=0, v_0=0$):
   $$\mathbf{\hat{m}}_t = \frac{\mathbf{m}_t}{1 - \beta_1^t}, \quad \mathbf{\hat{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_2^t}$$
4. Parameter Update:
   $$\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta}{\sqrt{\mathbf{\hat{v}}_t} + \epsilon} \mathbf{\hat{m}}_t$$

---

## 5. AdamW: Decoupled Weight Decay (Loshchilov & Hutter, 2019)

### The Flaw in standard Adam + L2 Regularization:
In standard SGD, adding an $L_2$ penalty $\frac{1}{2}\lambda \|\mathbf{w}\|^2$ is identical to weight decay ($\mathbf{w} \leftarrow (1 - \eta \lambda)\mathbf{w}$).
However, in Adam, the $L_2$ gradient $\lambda \mathbf{w}$ is divided by $\sqrt{\mathbf{v}_t}$! Parameters with large gradient variances receive **less weight decay**, violating the Bayesian prior.

### The AdamW Fix:
Decouple weight decay completely from the adaptive gradient moment updates:
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \lambda \mathbf{w}_t - \frac{\eta}{\sqrt{\mathbf{\hat{v}}_t} + \epsilon} \mathbf{\hat{m}}_t$$

**AdamW is the undisputed industry standard optimizer used to train GPT-4, LLaMA, Claude, and modern Transformers!**


In [ ]:
class AdamWOptimizer:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.wd = weight_decay
        self.m = np.zeros_like(params)
        self.v = np.zeros_like(params)
        self.t = 0
        
    def step(self, grad):
        self.t += 1
        # 1. Update biased 1st and 2nd moments
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * (grad**2)
        
        # 2. Bias correction
        m_hat = self.m / (1 - self.beta1**self.t)
        v_hat = self.v / (1 - self.beta2**self.t)
        
        # 3. Decoupled weight decay step
        self.params -= self.lr * self.wd * self.params
        
        # 4. Adaptive gradient update
        self.params -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return self.params

# Test AdamW
init_weights = np.array([2.0, -1.5])
opt = AdamWOptimizer(init_weights, lr=0.1)

for step in range(5):
    # Dummy gradient
    grad = np.array([0.4, -0.3])
    updated_w = opt.step(grad)
    print(f"Step {step+1}: Weights = {np.round(updated_w, 4)}")


---

## 6. Learning Rate Schedulers

A fixed learning rate $\eta$ is rarely optimal. Schedulers dynamically adjust $\eta(t)$ during training:

1. **Step Decay**: $\eta_t = \eta_0 \cdot \gamma^{\lfloor t / S \rfloor}$
2. **Exponential Decay**: $\eta_t = \eta_0 \cdot e^{-k t}$
3. **Cosine Annealing with Warmup** (Standard for Transformers):
   - **Linear Warmup**: Linearly increase $\eta$ from 0 to $\eta_{max}$ for the first $T_{warmup}$ steps.
   - **Cosine Decay**: Gradually decay following a half cosine wave:
     $$\eta_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})\left(1 + \cos\left(\frac{t - T_{warmup}}{T_{total} - T_{warmup}} \pi\right)\right)$$


In [ ]:
# Visualizing Cosine Annealing with Warmup
total_steps = 1000
warmup_steps = 100
lr_max = 1e-3
lr_min = 1e-5

lrs = []
for t in range(total_steps):
    if t < warmup_steps:
        # Linear warmup
        lr = lr_max * (t / warmup_steps)
    else:
        # Cosine decay
        progress = (t - warmup_steps) / (total_steps - warmup_steps)
        lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))
    lrs.append(lr)

plt.figure(figsize=(9, 4.5))
plt.plot(lrs, 'purple', linewidth=2.5)
plt.title("Cosine Annealing Learning Rate Schedule with Warmup")
plt.xlabel("Training Step"); plt.ylabel("Learning Rate")
plt.grid(True, alpha=0.3)
plt.show()


---

## 7. Summary & Key Takeaways

1. **SGD with Momentum** uses velocity to dampen ravines and accelerate convergence.
2. **RMSprop** adapts step sizes individually for each parameter using squared gradient histories.
3. **Adam** combines momentum and RMSprop with bias correction.
4. **AdamW** decouples weight decay from adaptive scaling, forming the foundation of modern Transformer training.
